# Exercise 4, the zero-dimensional energy balance model

These exercises go with **Lecture 4 &mdash; A zero-dimensional energy balance model**.
They reuse its `EBM` class and the SSP CO$_2$ data, reproduced in the setup cell, and
end with the classic *policy under uncertainty* problem: how the uncertainty in one
feedback parameter propagates into the projected warming.

Fill only the cells marked

```python
# ==== YOUR CODE ====
```


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


## Setup (given &mdash; just run it)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- parameters from Lecture 4 ---
S      = 1368.0        # solar constant [W/m2]
ALPHA  = 0.3           # albedo [-]
T0     = 14.0          # pre-industrial temperature [degC]
B      = -1.3          # climate feedback parameter [W/m2/degC]
a_co2  = 5.0           # CO2 forcing coefficient [W/m2]
C_heat = 51.0          # heat capacity [J/m2/degC]
A      = S * (1 - ALPHA) / 4 + B * T0     # = 221.2  (tuned so dT/dt = 0 at T0, 280 ppm)
CO2_PI = 280.0


class EBM:
    """Zero-order energy balance model (Lecture 4). Step it with run_model()."""

    def __init__(self, T, t, deltat, CO2, C=C_heat, a=a_co2, A=A, B=B,
                 CO2_PI=CO2_PI, alpha=ALPHA, S=S):
        self.T = np.array(T, dtype=float)
        self.t = t
        self.deltat = deltat
        self.C, self.a, self.A, self.B = C, a, A, B
        self.co2_pi, self.alpha, self.S = CO2_PI, alpha, S
        self.co2 = CO2

    def absorbed_solar_radiation(self, S, alpha):
        return S * (1 - alpha) / 4

    def outgoing_thermal_radiation(self, T, A, B):
        return A - B * T

    def greenhouse_effect(self, CO2, a, CO2_PI):
        return a * np.log(CO2 / CO2_PI)

    def tendency(self):
        T = self.T[-1] if self.T.size > 1 else self.T
        t = self.t[-1] if self.T.size > 1 else self.t
        return 1.0 / self.C * (
            self.absorbed_solar_radiation(self.S, self.alpha)
            - self.outgoing_thermal_radiation(T, self.A, self.B)
            + self.greenhouse_effect(self.co2(t), self.a, self.co2_pi))

    def step(self):
        newT = (self.T[-1] if self.T.size > 1 else self.T) + self.deltat * self.tendency()
        newt = (self.t[-1] if self.T.size > 1 else self.t) + self.deltat
        self.T = np.append(self.T, newT)
        self.t = np.append(self.t, newt)


def run_model(model, years):
    for _ in range(years):
        model.step()
    return model


# --- SSP CO2 pathways (Lecture 4) ---
co2_table = pd.read_csv("data/ssp_co2_concentrations.csv", comment="#")
def co2_from_table(column):
    years = co2_table["year"].to_numpy(float)
    conc = co2_table[column].to_numpy(float)
    return lambda t: np.interp(t, years, conc)

const_PI = lambda t: 280.0
print("setup ready. equilibrium check:",
      round(run_model(EBM(14.0, 0, 1.0, const_PI), 300).T[-1], 3), "degC (should stay ~14)")


## Exercise 1 &mdash; equilibrium climate sensitivity (ECS)

**ECS** is the eventual global warming after CO$_2$ is doubled and the system has
re-equilibrated. For this model it has a closed form. At the new equilibrium the
tendency is zero and the albedo has not changed, so

$$0 = -B\,\Delta T_\text{eq} + a\ln 2
\qquad\Longrightarrow\qquad
\boxed{\ \text{ECS} = -\dfrac{a\ln 2}{B}\ }$$

**Your task.**
1. Write `ecs(a, B)` for the formula above.
2. Run an **abrupt 2×CO$_2$** experiment: start at equilibrium, switch CO$_2$ to
   $2\times280$ ppm, integrate 300 years, and check $\Delta T = T(t) - T_0$ approaches
   `ecs(a_co2, B)`.
3. Plot ECS as a function of $B$ over $B \in [-2.0, -0.2]$.

**Hints.**
* `ecs(a, B)` is one line: `-a * np.log(2) / B`.
* Abrupt 2×CO$_2$: `EBM(T0, 0, 1.0, lambda t: 2*280.0)` then `run_model(m, 300)`.
* For the ECS-vs-$B$ curve, `Bvec = np.linspace(-2.0, -0.2, 100)`.


In [ ]:
# ==== YOUR CODE ====
def ecs(a, B):
    return None                    # <-- -a * np.log(2) / B

m = None                           # <-- EBM(T0, 0, 1.0, lambda t: 2*280.0);  run_model(m, 300)
# ===================

if m is not None:
    dT = m.T - m.T[0]
    print(f"model dT after 300 yr : {dT[-1]:.2f} degC")
    print(f"analytic ECS          : {ecs(a_co2, B):.2f} degC")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
if m is not None:
    ax[0].plot(m.t, m.T - m.T[0], color="tab:red", label="$\\Delta T(t)$")
    ax[0].axhline(ecs(a_co2, B), color="darkred", ls="--", label="ECS (analytic)")
    ax[0].set_xlabel("year"); ax[0].set_ylabel("$\\Delta T$ [degC]")
    ax[0].set_title("abrupt 2xCO$_2$"); ax[0].legend(); ax[0].grid(alpha=0.3)

Bvec = np.linspace(-2.0, -0.2, 100)
if ecs(a_co2, -1.3) is not None:
    ax[1].plot(Bvec, ecs(a_co2, Bvec))
    ax[1].axvline(-1.3, color="0.5", ls=":")
    ax[1].set_xlabel("feedback parameter B [W/m$^2$/degC]"); ax[1].set_ylabel("ECS [degC]")
    ax[1].set_title("ECS diverges as B -> 0"); ax[1].grid(alpha=0.3)
    ax[1].set_ylim(0, 15)
fig.tight_layout()


```{admonition} Answers
:class: note
1. What is the ECS for $B = -1.3$? How long does the model take to reach ~63% of it,
   and what sets that timescale (which term in the equation)?
2. The ECS&ndash;$B$ curve blows up as $B \to 0$. What does $B \ge 0$ mean physically,
   and why is it called a *runaway*?
3. The IPCC "likely" ECS range is about 2.5&ndash;4 °C. What range of $B$ does that
   correspond to in this model?
```


## Exercise 2 &mdash; uncertainty in $B$, and what it does to the projection

$B$ is not a knob we set &mdash; it is an emergent property of the real climate, and it
is genuinely uncertain. A widely used estimate is $B = -1.3 \pm 0.4$ W m⁻² K⁻¹
(1$\sigma$). Here you propagate that uncertainty through to ECS and to a 2100
projection. This is the heart of *policy under uncertainty*.

**Your task.**
1. Draw 50 000 samples $B \sim \mathcal N(-1.3,\, 0.4^2)$. Histogram them.
2. Map each sample through `ecs(a_co2, B)` to get an ECS distribution. Histogram it.
   Drop nonphysical samples ($B \ge 0$, or ECS outside 0&ndash;20 °C).
3. Compare **`ecs(a_co2, mean(B))`** with **`mean(ecs(a_co2, B))`**, and report
   $P(\text{ECS} > 4\ °\text{C})$.

**Hints.**
* `rng = np.random.default_rng(0); Bs = rng.normal(-1.3, 0.4, 50_000)`.
* `Bs = Bs[Bs < 0]` before mapping to ECS.
* `E = ecs(a_co2, Bs); E = E[(E > 0) & (E < 20)]`.
* `np.mean(E > 4)` is the exceedance probability.


In [ ]:
rng = np.random.default_rng(0)

# ==== YOUR CODE ====
Bs = None            # <-- 50_000 normal samples, then keep only Bs < 0
E  = None            # <-- ecs(a_co2, Bs), then keep 0 < E < 20

ecs_of_meanB = None  # <-- ecs(a_co2, Bs.mean())
mean_ecs     = None  # <-- E.mean()
p_gt_4       = None  # <-- np.mean(E > 4)
# ===================

print(f"ECS( mean B )      = {ecs_of_meanB}")
print(f"mean( ECS(B) )     = {mean_ecs}")
print(f"P(ECS > 4 degC)    = {p_gt_4}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
if Bs is not None:
    ax[0].hist(Bs, bins=60, color="tab:blue", alpha=0.8)
    ax[0].set_xlabel("B [W/m$^2$/degC]"); ax[0].set_ylabel("samples")
    ax[0].set_title("input: B is Gaussian")
if E is not None:
    ax[1].hist(E, bins=60, color="tab:red", alpha=0.8)
    ax[1].axvline(mean_ecs, color="k", label=f"mean ECS(B) = {mean_ecs:.1f}")
    ax[1].axvline(ecs_of_meanB, color="k", ls="--", label=f"ECS(mean B) = {ecs_of_meanB:.1f}")
    ax[1].set_xlabel("ECS [degC]"); ax[1].set_title("output: ECS is skewed")
    ax[1].legend(fontsize=8)
fig.tight_layout()


```{admonition} Answers
:class: note
1. The input $B$ is symmetric but the output ECS is not. Which direction is the skew,
   and why does the $-1/B$ shape produce it?
2. Is `mean(ECS(B))` bigger or smaller than `ECS(mean B)`? Does accounting for the
   uncertainty make the *expected* warming better or worse news?
3. $P(\text{ECS} > 4\ °\text{C})$ is the probability of an outcome usually called
   "high-end". A policy that plans only for the central ECS &mdash; what risk is it
   ignoring?
```


## Exercise 3 &mdash; add a temperature-dependent albedo (bridge to Lecture 5)

So far the albedo was fixed. If a colder planet is icier, and ice is bright, then
$\alpha$ should *increase* as $T$ falls &mdash; the **ice&ndash;albedo feedback**. Add it
and watch the model gain a second stable state.

**Your task.**
1. The step-shaped `calc_alpha(T)` is given (complete).
2. Complete `EBM_ice`, a subclass whose `absorbed_solar_radiation` uses
   `calc_alpha(current_T)` instead of the fixed `self.alpha`.
3. Run the model from many starting temperatures (given loop) and see which ones end
   warm and which end frozen.

**Hints.**
* In the subclass method, get the current temperature with
  `T = self.T[-1] if self.T.size > 1 else self.T`, then
  `return self.S * (1 - calc_alpha(T)) / 4`.
* `calc_alpha` needs a scalar; the loop already runs one model at a time.


In [ ]:
def calc_alpha(T, alpha0=0.3, alphai=0.5, dT=10.0):
    """Albedo rises from alpha0 (ice-free) to alphai (frozen) as T falls (Lecture 5)."""
    if T < -dT:
        return alphai
    if T >= dT:
        return alpha0
    return alphai + (alpha0 - alphai) * (T + dT) / (2 * dT)


class EBM_ice(EBM):
    def absorbed_solar_radiation(self, S, alpha):
        # ==== YOUR CODE ====
        # use calc_alpha(current_T) instead of the passed-in `alpha`
        T = self.T[-1] if self.T.size > 1 else self.T
        return S * (1 - alpha) / 4          # <-- replace `alpha` with calc_alpha(T)
        # ===================


starts = np.arange(-60, 40, 5.0)
finals = []
for Tstart in starts:
    m = run_model(EBM_ice(float(Tstart), 0, 1.0, const_PI), 400)
    finals.append(m.T[-1])
finals = np.array(finals)
print("final temperatures:", np.round(finals, 1))
print("distinct end states:", np.round(np.unique(finals.round(1)), 1))


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(8, 5))
for Tstart in starts:
    m = run_model(EBM_ice(float(Tstart), 0, 1.0, const_PI), 400)
    ax.plot(m.t, m.T, lw=1)
ax.set_xlabel("year"); ax.set_ylabel("temperature [degC]")
ax.set_title("Ice-albedo EBM from many starting temperatures")
ax.axhspan(-60, -10, color="lightblue", alpha=0.3)
ax.axhspan(10, 40, color="red", alpha=0.08)
ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. How many distinct final temperatures are there now, and roughly what are they?
   (Compare with the *single* equilibrium of the constant-albedo model.)
2. Which starting temperatures end frozen? Is there a sharp boundary between "ends warm"
   and "ends frozen"?
3. This is the model of Lecture 5. What did adding *one* temperature-dependent term do
   to the number of climates the planet can have?
```


## Exercise 4 &mdash; transient warming, equilibrium warming, and what is "in the pipeline"

The ECS of Exercise 1 is the *eventual* warming. At any moment while CO$_2$ is still
rising, the actual (**transient**) warming is *less* than the equilibrium value for the
current CO$_2$, because the heat-capacity term `C` makes temperature lag the forcing.
The gap is the warming "in the pipeline".

**Your task.**
1. Run the EBM under **SSP5-8.5** (the high scenario) from 1850, and also compute, for
   each year, the *equilibrium* $\Delta T$ for that year's CO$_2$:
   $\Delta T_\text{eq}(t) = a\ln\!\big(\text{CO}_2(t)/280\big) / (-B)$.
2. Plot the two curves; the vertical gap is committed warming.
3. **Freeze** CO$_2$ at its 2100 value and keep integrating to 2300. Watch $\Delta T$
   climb to close the gap.

**Hints.**
* `ssp585 = co2_from_table("SSP5-8.5")`; `m = run_model(EBM(T0, 1850, 1.0, ssp585), 250)`.
* `dT_eq = a_co2 * np.log(ssp585(m.t) / 280.0) / (-B)` &mdash; `m.t` is an array,
  `ssp585` is vectorised.
* For the freeze: `c2100 = float(ssp585(2100)); frozen = lambda t: c2100`; start a
  fresh model from the 2100 state: `EBM(m.T[np.argmin(abs(m.t-2100))], 2100, 1.0, frozen)`.


In [ ]:
ssp585 = co2_from_table("SSP5-8.5")

# ==== YOUR CODE ====
m = None            # <-- run_model(EBM(T0, 1850, 1.0, ssp585), 250)   -> reaches 2100
dT_eq = None        # <-- a_co2 * np.log(ssp585(m.t) / 280.0) / (-B)

# freeze CO2 at the 2100 level and run 200 more years from the 2100 temperature
c2100 = float(ssp585(2100))
T_2100 = None       # <-- m.T[np.argmin(np.abs(m.t - 2100))]
m_frozen = None     # <-- run_model(EBM(T_2100, 2100, 1.0, lambda t: c2100), 200)
# ===================

if m is not None:
    gap_2100 = dT_eq[-1] - (m.T[-1] - T0)
    print(f"2100: transient dT = {m.T[-1]-T0:.2f},  equilibrium dT = {dT_eq[-1]:.2f}")
    print(f"      committed (pipeline) warming at 2100 = {gap_2100:.2f} degC")
if m_frozen is not None:
    print(f"2300 (CO2 frozen at 2100 level): dT = {m_frozen.T[-1]-T0:.2f} degC")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(10, 4.5))
if m is not None:
    ax.plot(m.t, m.T - T0, color="tab:red", lw=2, label="transient $\\Delta T$ (EBM, SSP5-8.5)")
    ax.plot(m.t, dT_eq, color="darkred", ls="--", label="equilibrium $\\Delta T$ for that year's CO$_2$")
    ax.fill_between(m.t, m.T - T0, dT_eq, color="orange", alpha=0.25, label="warming in the pipeline")
if m_frozen is not None:
    ax.plot(m_frozen.t, m_frozen.T - T0, color="tab:blue", lw=2,
            label="CO$_2$ frozen at 2100 level")
ax.set_xlabel("year"); ax.set_ylabel("$\\Delta T$ [degC]")
ax.set_title("Transient vs equilibrium warming")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. What is the committed ("pipeline") warming in 2100 under SSP5-8.5 &mdash; the gap
   between the transient and equilibrium curves? Which single term in the model creates
   it?
2. After CO$_2$ is frozen at the 2100 level, how long does it take $\Delta T$ to get
   within 0.1 °C of the equilibrium value? Compare with $\tau = C/(-B)$.
3. The real climate's transient-to-equilibrium ratio (TCR/ECS) is about 0.5&ndash;0.7.
   Is this model's ratio in that range? A real ocean mixes heat into the deep &mdash;
   would that push the ratio up or down?
```


## Where this goes next

* Exercise 3 is the whole of **Lecture 5** (Snowball Earth) &mdash; and its own exercise
  set.
* The uncertainty propagation of Exercises 1&ndash;2 is why climate projections are
  always given as *ranges*; **Module 4** does it with full CMIP6 ensembles.

```{note} Sources
Combines the exercises of Lecture 4 with the *policy goals under uncertainty* problem
from the [*Climate of the Ocean*](https://github.com/florianboergel/climateoftheocean)
course (after Henri Drake, MIT 18.S191), for **CE524 Applied Hydroclimatology**.
Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
